In [ ]:
import tensorflow_datasets as tfds

(train, test, validation), info = tfds.load(
    "speech_commands",
    split=["train", "test", "validation"],
    with_info=True
)
print(len(train))
print(len(test))
print(len(validation))
print("=== Info general ===")
print(info)
print("=== Nombres de las etiquetas ===")
print(info.features['label'].names)
print("=== Cantidad de ejemplos por etiqueta en cada conjunto ===")
trainElements = [0] * 12
testElements = [0] * 12
valElements = [0] * 12

for e in train:
    trainElements[e['label'].numpy()] += 1

for e in test:
    testElements[e['label'].numpy()] += 1

for e in validation:
    valElements[e['label'].numpy()] += 1

print(trainElements)
print(testElements)
print(valElements)


In [ ]:
import IPython.display as ipd
import tensorflow as tf
import gc
def add_noise(audio, noise_factor=0.05):
    audio = tf.cast(audio, tf.float32)
    audio_power = tf.reduce_mean(tf.square(audio))
    noise = tf.random.normal(shape=tf.shape(audio), mean=0.0, stddev=1.0)
    noise_power = tf.reduce_mean(tf.square(noise))
    scale = tf.sqrt(audio_power / noise_power) * noise_factor
    return audio + scale * noise

import numpy as np
from scipy import signal as scipy_signal

def pitch_shift(audio, min_semitones=-3, max_semitones=3):
    audio_np = audio.numpy() if hasattr(audio, 'numpy') else np.array(audio)
    audio_np = audio_np.astype(np.float32)
    
    semitones = np.random.uniform(min_semitones, max_semitones)
    factor = 2.0 ** (semitones / 12.0)
    
    original_length = len(audio_np)
    new_length = int(original_length / factor)
    
    resampled = scipy_signal.resample(audio_np, new_length)
    
    if new_length > original_length:
        result = resampled[:original_length]
    else:
        result = np.pad(resampled, (0, original_length - new_length))
    
    return tf.convert_to_tensor(result, dtype=tf.float32)

def change_volume(audio, min_gain=0.1, max_gain=5.0):
    audio = tf.cast(audio, tf.float32)
    gain = tf.random.uniform([], min_gain, max_gain)
    return audio * gain

def time_shift(audio, shift_max=24000): 
    audio = tf.cast(audio, tf.float32)
    shift = tf.random.uniform([], -shift_max, shift_max, dtype=tf.int32)
    return tf.roll(audio, shift, axis=0)

def noise(audio1):
    audio = audio1['audio']
    label = audio1['label']

    audio = add_noise(audio, noise_factor=0.3)

    return {'audio': audio, 'label': label}

def volume(example):
    audio = example['audio']
    label = example['label']

    audio = change_volume(audio)

    return {'audio': audio, 'label': label}

def pitch(example):
    audio = example['audio']
    label = example['label']

    audio = tf.py_function(func=pitch_shift, inp=[audio], Tout=tf.float32)
    audio.set_shape([None])

    return {'audio': audio, 'label': label}


def shift(example):
    audio = example['audio']
    label = example['label']

    audio = time_shift(audio)

    return {'audio': audio, 'label': label}

for e in train.take(1):
    audio = e['audio']
    label = e['label']

for audio in train.take(1):
    if (audio['label'].numpy() == 4):
        original_audio = audio['audio'].numpy()
        
        print("Original:")
        display(ipd.Audio(original_audio, rate=16000))
        
        print("Time Shift:")
        shifted = shift(audio)["audio"].numpy()
        display(ipd.Audio(shifted, rate=16000))
        
        print("Noise:")
        noisy = noise(audio)["audio"].numpy()
        display(ipd.Audio(noisy, rate=16000))
        
        print("Volume:")
        vol_changed = volume(audio)["audio"].numpy()
        display(ipd.Audio(vol_changed, rate=16000))

        print("Pitch:")
        pitchC = pitch(audio)["audio"].numpy()
        display(ipd.Audio(pitchC, rate=16000))
        
        break
        

for audio in train.take(1):
    if (audio['label'].numpy() == 4):
        display(ipd.Audio(audio['audio'].numpy(), rate=16000))
        display(ipd.Audio(shift(audio)["audio"].numpy(), rate=16000))
        display(ipd.Audio(noise(audio)["audio"].numpy(), rate=16000))
        display(ipd.Audio(volume(audio)["audio"].numpy(), rate=16000))
        break
def to_float(example):
    return {
        'audio': tf.cast(example['audio'], tf.float32),
        'label': example['label']
    }

train = train.map(to_float)
train_noise = train.map(noise)
train_volume = train.map(volume)
train_shift = train.map(shift)
train_pitch = train.map(pitch) 
train = train.concatenate(train_noise).concatenate(train_volume).concatenate(train_shift).concatenate(train_pitch)
del train_noise
del train_volume
del train_shift
del train_pitch
gc.collect()
print(len(train))

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
print("GPUs disponibles:", tf.config.list_physical_devices('GPU'))
def getSpectrogram(audio):

    audio = tf.cast(audio, tf.float32) / 32768.0
    rightSamples = 16000
    audio = audio[:rightSamples]
    audio = tf.pad(audio, [[0, rightSamples - tf.shape(audio)[0]]])
    
    stft = tf.signal.stft(
        audio,
        frame_length=640,
        frame_step=320
    )

    spectrogram = tf.abs(stft)

    num_spectrogram_bins = spectrogram.shape[-1]
    num_mel_bins = 40
    sample_rate = 16000

    mel_weight_matrix = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins,
        num_spectrogram_bins,
        sample_rate
    )

    mel_spectrogram = tf.tensordot(
        spectrogram,
        mel_weight_matrix,
        1
    )

    mel_spectrogram.set_shape(
        spectrogram.shape[:-1].concatenate([num_mel_bins])
    )

    log_mel_spectrogram = tf.math.log(mel_spectrogram + 1e-6)
    log_mel_spectrogram = log_mel_spectrogram[..., tf.newaxis]
    log_mel_spectrogram.set_shape((49, 40, 1))
    return log_mel_spectrogram

def showSpectrogram(spectrogram):
    spectrogram = tf.squeeze(spectrogram)
    plt.figure(figsize=(8,4))
    plt.imshow(tf.transpose(spectrogram), aspect='auto', origin='lower')
    plt.colorbar()
    plt.title("Spectrogram")
    plt.xlabel("Time")
    plt.ylabel("Frequency")
    plt.show()
i = 0
for audio in train:
    if (audio["label"].numpy() == 4):
        print(audio['audio'])
        showSpectrogram(getSpectrogram(audio['audio']))
        i += 1
        if i == 10:
            break
    

In [ ]:
def spec_augment(spectrogram, 
                 time_mask_param=10,      
                 freq_mask_param=8,      
                 n_time_masks=2,          
                 n_freq_masks=2):        

    spec = tf.identity(spectrogram)  
    
    for i in range(n_time_masks):
        t = tf.random.uniform([], 0, time_mask_param, dtype=tf.int32)
        t0 = tf.random.uniform([], 0, 49 - t, dtype=tf.int32)
        
        mask = tf.concat([
            tf.ones([t0, 40, 1]),
            tf.zeros([t, 40, 1]),
            tf.ones([49 - t0 - t, 40, 1])
        ], axis=0)
        
        spec = spec * mask
    
    for i in range(n_freq_masks):
        f = tf.random.uniform([], 0, freq_mask_param, dtype=tf.int32)
        f0 = tf.random.uniform([], 0, 40 - f, dtype=tf.int32)
        
        mask = tf.concat([
            tf.ones([49, f0, 1]),
            tf.zeros([49, f, 1]),
            tf.ones([49, 40 - f0 - f, 1])
        ], axis=1)
        
        spec = spec * mask
    
    return spec

def preprocess_with_augment(spec, label):
    spec = spec_augment(spec)
    
    return spec, label

def prepareData(e):
    audio = getSpectrogram(e["audio"])
    label = e["label"]
    return audio, label

unknown = train.filter(lambda x: x["label"] == 11)
known = train.filter(lambda x: x["label"] != 11)
unknown = unknown.shuffle(430000).take(18000)
count = unknown.reduce(0, lambda x, _: x + 1)
print("Unknown finales:", count.numpy())
train = known.concatenate(unknown)
train = train.shuffle(430000)
del known
del unknown

train1 = train.map(prepareData)
train1 = train1.map(preprocess_with_augment)
train1 = train1.batch(32)
validation1 = validation.map(prepareData).batch(32)
test1 = test.map(prepareData).batch(32)
del train
del validation
del test
gc.collect()

In [ ]:
import tensorflow as tf
class PatchEmbedding(tf.keras.layers.Layer):
    def __init__(self, patch_size, embed_dim):
        super().__init__()
        self.patch_h, self.patch_w = patch_size
        self.proj = tf.keras.layers.Dense(embed_dim)

    def call(self, images):
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_h, self.patch_w, 1],
            strides=[1, self.patch_h, self.patch_w, 1],
            rates=[1, 1, 1, 1],
            padding='VALID'
        )
        patches = tf.reshape(patches, [tf.shape(images)[0], -1, patches.shape[-1]])
        return self.proj(patches)

class AddPositionEmbedding(tf.keras.layers.Layer):
    def __init__(self, embed_dim, max_len=500):
        super().__init__()
        self.pos_emb = self.add_weight(
            shape=(1, max_len, embed_dim),
            initializer="random_normal"
        )
        
    def call(self, x):
        seq_len = tf.shape(x)[1]
        return x + self.pos_emb[:, :seq_len, :]
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.norm1 = tf.keras.layers.LayerNormalization()
        self.attn = tf.keras.layers.MultiHeadAttention(num_heads, key_dim=embed_dim // num_heads, dropout=dropout)
        self.norm2 = tf.keras.layers.LayerNormalization()

        self.mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(mlp_dim, activation='gelu'),
            tf.keras.layers.Dropout(dropout),
            tf.keras.layers.Dense(embed_dim)
        ])

    def call(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x))
        x = x + self.mlp(self.norm2(x)) 
        return x
        
class VisionTransformer(tf.keras.Model):
    def __init__(self, num_classes, patch_size,
                 embed_dim=128, depth=6, num_heads=2, mlp_dim=256):
        super().__init__()

        self.patch_embed = PatchEmbedding(patch_size, embed_dim)
        self.pos_embed = AddPositionEmbedding(
            embed_dim=embed_dim
        )
        self.embed_dropout = tf.keras.layers.Dropout(0.1)

        self.transformer = [
            TransformerBlock(embed_dim, num_heads, mlp_dim)
            for _ in range(depth)
        ]

        self.norm = tf.keras.layers.LayerNormalization()
        self.head = tf.keras.layers.Dense(num_classes, activation='softmax')

    def call(self, x):
        x = self.patch_embed(x)
        x = self.pos_embed(x)
        x = self.embed_dropout(x, training=False)
        for block in self.transformer:
            x = block(x)

        x = self.norm(x)
        x = tf.reduce_mean(x, axis=1)
        return self.head(x)
        


model = VisionTransformer(
    num_classes=12,  
    patch_size=(4, 8),
    embed_dim=192,
    depth=8,
    num_heads=3,
    mlp_dim=384
)

"""
  tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7
    ),"""
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=7,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath='best_model.keras',  # dónde guardar
        monitor='val_accuracy',       # qué métrica vigilar
        save_best_only=True,          # solo guarda si mejora
        verbose=1
    )
]


model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    train1,
    validation_data=validation1,
    epochs=35,
    callbacks=callbacks
)
model.save("0.9_6000_dropout0.3_.keras")
model.evaluate(test1)